# PathSnap model training notebook

This notebook trains one candidate-classification model for the PathSnap task.

**Important:** The PathSnap documents require the 80 training trips to be used for model fitting/cross-validation and the 10 student-testing trips for development-stage comparison. Evaluator-testing data must not influence feature selection, model selection, hyperparameters, or thresholds.

This notebook deliberately does **not** invent dataset-specific column names. Set `DATA_PATH`, `TARGET_COLUMN`, and `GROUP_COLUMN` below after inspecting the team's actual feature-engineered dataset.


## Model: LightGBM

The model is evaluated using:
- Group-aware cross-validation by trip
- Observation-level Top-1 Accuracy
- Macro F1
- Log Loss where probabilities are available
- Training and inference time

The notebook saves the fitted model and a compact experiment result file. Actual graphs/results are generated only after the notebook is run on the team's real data.

In [ ]:
# =========================
# 1. Configuration
# =========================

from pathlib import Path
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix,
)



# ---- CHANGE THESE ----
DATA_PATH = Path("../data/processed/features.csv")
TARGET_COLUMN = "target_arc_id"     # Ground-truth road-segment/candidate label
GROUP_COLUMN = "trip_id"            # Trip identifier; MUST NOT be used as a feature

# Optional: if your feature table already contains a split column, set its name here.
SPLIT_COLUMN = "partition"

# Expected values if SPLIT_COLUMN exists:
TRAIN_SPLIT_VALUE = "train"
STUDENT_TEST_SPLIT_VALUE = "student_test"

N_SPLITS = 5
RANDOM_STATE = 42

ARTIFACT_DIR = Path("../artifacts/lightgbm")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_PATH.resolve())
print("Target:", TARGET_COLUMN)
print("Group:", GROUP_COLUMN)

In [ ]:
# =========================
# 2. Load and inspect data
# =========================

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(20))

In [ ]:
# =========================
# 3. Select permitted training data
# =========================

required = {TARGET_COLUMN, GROUP_COLUMN}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if SPLIT_COLUMN in df.columns:
    train_df = df[df[SPLIT_COLUMN] == TRAIN_SPLIT_VALUE].copy()
    student_df = df[df[SPLIT_COLUMN] == STUDENT_TEST_SPLIT_VALUE].copy()
else:
    # If the feature table does not contain partitions, use ONLY the training
    # feature table here. Do not mix evaluator-testing rows into this notebook.
    train_df = df.copy()
    student_df = None

print("Training rows:", len(train_df))
print("Training trips:", train_df[GROUP_COLUMN].nunique())

if student_df is not None:
    print("Student-test rows:", len(student_df))
    print("Student-test trips:", student_df[GROUP_COLUMN].nunique())

In [ ]:
# =========================
# 4. Build X/y and prevent leakage
# =========================

# Keep only numeric model inputs.
# Target and trip identifiers are explicitly excluded.
excluded = {TARGET_COLUMN, GROUP_COLUMN}
if SPLIT_COLUMN in train_df.columns:
    excluded.add(SPLIT_COLUMN)

feature_columns = [
    c for c in train_df.columns
    if c not in excluded and pd.api.types.is_numeric_dtype(train_df[c])
]

if not feature_columns:
    raise ValueError("No numeric feature columns found. Check your feature-engineering output.")

X = train_df[feature_columns].copy()
y = train_df[TARGET_COLUMN].copy()
groups = train_df[GROUP_COLUMN].copy()

print(f"Number of features: {len(feature_columns)}")
print(feature_columns)

# Tree models can handle different feature scales, so no standardisation is required.
# Replace this with your team's actual preprocessing if needed.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

print("\nNumber of classes:", y.nunique())
print("Class distribution:")
display(y.value_counts().head(20))

In [ ]:
# =========================
# 5. Model definition
# =========================

from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

print(model)

In [ ]:
# =========================
# 6. Group-aware cross-validation
# =========================

gkf = GroupKFold(n_splits=N_SPLITS)

fold_rows = []
oof_pred = np.empty(len(y), dtype=object)
oof_proba = None

classes = np.sort(y.unique())
class_to_index = {c: i for i, c in enumerate(classes)}

if hasattr(model, "predict_proba"):
    oof_proba = np.zeros((len(y), len(classes)), dtype=float)

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    fold_model = model
    start = time.perf_counter()
    fold_model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    pred = fold_model.predict(X_va)
    inference_time = time.perf_counter() - start

    oof_pred[va_idx] = pred

    row = {{
        "fold": fold,
        "train_time_s": train_time,
        "inference_time_s": inference_time,
        "accuracy": accuracy_score(y_va, pred),
        "macro_f1": f1_score(y_va, pred, average="macro", zero_division=0),
    }}

    if hasattr(fold_model, "predict_proba"):
        proba = fold_model.predict_proba(X_va)
        fold_classes = list(fold_model.classes_)
        for j, c in enumerate(fold_classes):
            oof_proba[va_idx, class_to_index[c]] = proba[:, j]
        row["log_loss"] = log_loss(
            y_va, oof_proba[va_idx], labels=classes
        )

    fold_rows.append(row)

cv_results = pd.DataFrame(fold_rows)

print("Fold results:")
display(cv_results)

print("\nCV mean:")
display(cv_results.mean(numeric_only=True).to_frame("mean"))

In [ ]:
# =========================
# 7. Cross-validation diagnostics
# =========================

print("OOF Accuracy:", accuracy_score(y, oof_pred))
print("OOF Macro F1:", f1_score(y, oof_pred, average="macro", zero_division=0))

if oof_proba is not None:
    print("OOF Log Loss:", log_loss(y, oof_proba, labels=classes))

plt.figure(figsize=(7, 4))
plt.bar(
    np.arange(1, len(cv_results) + 1),
    cv_results["macro_f1"]
)
plt.xlabel("CV fold")
plt.ylabel("Macro F1")
plt.title(f"{model_name}: group-aware CV Macro F1")
plt.xticks(np.arange(1, len(cv_results) + 1))
plt.tight_layout()
plt.show()

cm = confusion_matrix(y, oof_pred, labels=classes)

plt.figure(figsize=(8, 7))
plt.imshow(cm, aspect="auto")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(f"{model_name}: OOF confusion matrix")
plt.colorbar(label="Count")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 8. Fit final model on all permitted training trips
# =========================

final_model = model

start = time.perf_counter()
final_model.fit(X, y)
final_train_time = time.perf_counter() - start

print(f"Final training time: {final_train_time:.3f} s")

In [ ]:
# =========================
# 9. Student-testing evaluation
# =========================

student_results = {}

if student_df is None:
    print("Student-test rows are not present in this feature table.")
    print("Run this section after supplying the separate 10-trip student-testing feature table.")
else:
    X_student = student_df[feature_columns].copy()
    y_student = student_df[TARGET_COLUMN].copy()

    X_student = X_student.replace([np.inf, -np.inf], np.nan)
    X_student = X_student.fillna(X.median(numeric_only=True))

    start = time.perf_counter()
    student_pred = final_model.predict(X_student)
    inference_time = time.perf_counter() - start

    student_results["top1_accuracy"] = accuracy_score(y_student, student_pred)
    student_results["macro_f1"] = f1_score(
        y_student, student_pred, average="macro", zero_division=0
    )
    student_results["inference_time_s"] = inference_time

    if hasattr(final_model, "predict_proba"):
        student_proba = final_model.predict_proba(X_student)
        student_results["log_loss"] = log_loss(
            y_student,
            student_proba,
            labels=final_model.classes_
        )

    print("Student-test results:")
    display(pd.DataFrame([student_results]))

    print("\nClassification report:")
    print(classification_report(y_student, student_pred, zero_division=0))

In [ ]:
# =========================
# 10. Save model + experiment record
# =========================

joblib.dump(
    {
        "model": final_model,
        "feature_columns": feature_columns,
        "target_column": TARGET_COLUMN,
        "group_column": GROUP_COLUMN,
        "classes": list(getattr(final_model, "classes_", classes)),
    },
    ARTIFACT_DIR / "model.joblib"
)

experiment = {
    "model": model.__class__.__name__,
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "n_training_rows": int(len(train_df)),
    "n_training_trips": int(train_df[GROUP_COLUMN].nunique()),
    "n_features": int(len(feature_columns)),
    "cv_macro_f1_mean": float(cv_results["macro_f1"].mean()),
    "cv_macro_f1_std": float(cv_results["macro_f1"].std()),
    "cv_accuracy_mean": float(cv_results["accuracy"].mean()),
    "student_test": student_results,
    "feature_columns": feature_columns,
}

with open(ARTIFACT_DIR / "experiment.json", "w") as f:
    json.dump(experiment, f, indent=2)

cv_results.to_csv(ARTIFACT_DIR / "cv_results.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())

## Interpretation checklist

Before using this result in the report, record:

1. Why this model is suitable for PathSnap.
2. The important hyperparameters and their initial values.
3. The optimisation objective/loss.
4. CV Macro F1 and its fold-to-fold stability.
5. Student-testing Top-1 Accuracy, Macro F1 and Log Loss where applicable.
6. Training and inference time.
7. Evidence for overfitting or underfitting.
8. How the model's probability outputs will be consumed by the later HMM/Viterbi route reconstruction.

Do not write the final report numbers until this notebook has actually been executed on your team's data. The submission guidelines explicitly require actual experimental results and prohibit fabricated training curves/results.